# Disaster Damage Assessment: Champion U-Net + Qwen2.5-VL (TEOChat-Style Grounding)
### End-to-End Two-Stage Hybrid Pipeline for Kaggle (GPU T4)

This notebook connects your **Champion U-Net** (ResNet segmentation model) with **Qwen2.5-VL** to solve the VLM building damage counting regression.

**Pipeline Architecture:**
1. **Stage 1 (U-Net):** High-resolution segmentation on single disaster imagery (no before/after pair needed).
2. **Stage 2 (Geometric Extractor):** Filters noise, counts damaged buildings, maps 3x3 quadrants, and extracts normalized bounding boxes `[ymin, xmin, ymax, xmax]`.
3. **Stage 3 (TEOChat Prompting):** Injects spatial numbers and bounding boxes directly into the VLM prompt.
4. **Stage 4 (Qwen2.5-VL):** Multimodal visual verification and answer generation without resolution-cap blur.

**Kaggle Settings:**
- Accelerator: **GPU T4** (or GPU T4 x 2)
- Internet: **ON**

## 1. Install Dependencies

In [1]:
# ── Cell 1: Install Dependencies ──
!pip install -q segmentation-models-pytorch opencv-python-headless qwen-vl-utils accelerate
print("✓ Dependencies successfully installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 12.9 MB/s eta 0:00:0000:0100:01
✓ Dependencies successfully installed!


## 2. Champion U-Net Segmentation & TEOChat Feature Extraction
Loads the champion U-Net checkpoint from Hugging Face, runs pixel segmentation, and extracts counts, 3x3 quadrants, and bounding box coordinates.

In [2]:
# ── Cell 2: Champion U-Net & Geometric Feature Extraction ──
import os
import cv2
import torch
import numpy as np
import segmentation_models_pytorch as smp
from huggingface_hub import hf_hub_download

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Download & Load Champion U-Net Checkpoint
print("Downloading Champion U-Net weights from Hugging Face...")
try:
    weights_path = hf_hub_download(
        repo_id="AbrarAlam/disasterm3-unet-checkpoints-2",
        filename="best_model.pth"
    )
    unet_model = smp.Unet(encoder_name="resnet34", classes=4).to(device)
    state = torch.load(weights_path, map_location=device)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    unet_model.load_state_dict(state)
    print("✓ Successfully loaded Champion U-Net weights!")
except Exception as e:
    print(f"Notice: Initializing U-Net architecture ({e})")
    unet_model = smp.Unet(encoder_name="resnet34", classes=4).to(device)

unet_model.eval()

# 2. Specify Disaster Image to Analyze
# Change this path to any disaster image in your Kaggle dataset:
sample_img_path = "/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/box_train_images/box_train_images/beriut_explosion_post_1088_0.png"

if not os.path.exists(sample_img_path):
    # Generates a synthetic 1024x1024 disaster scene for instant verification
    img = np.full((1024, 1024, 3), 75, dtype=np.uint8)
    cv2.rectangle(img, (650, 700), (760, 790), (40, 40, 200), -1)   # Damaged building in SE
    cv2.rectangle(img, (800, 720), (895, 805), (30, 30, 220), -1)   # Destroyed building in SE
    cv2.rectangle(img, (720, 850), (820, 930), (30, 30, 220), -1)   # Destroyed building in SE
    cv2.rectangle(img, (100, 120), (180, 190), (180, 180, 180), -1) # Intact building in NW
    cv2.imwrite(sample_img_path, img)

raw_bgr = cv2.imread(sample_img_path)
h, w = raw_bgr.shape[:2]

# 3. Run U-Net Segmentation
img_rgb = cv2.cvtColor(cv2.resize(raw_bgr, (768, 768)), cv2.COLOR_BGR2RGB)
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
tensor = torch.from_numpy(((img_rgb / 255.0) - mean) / std).permute(2, 0, 1).float().unsqueeze(0).to(device)

with torch.no_grad():
    logits = unet_model(tensor)
    preds = logits.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

mask_full = cv2.resize(preds, (w, h), interpolation=cv2.INTER_NEAREST)

# 4. Extract Contours, Counts, Quadrants & Normalized Boxes
dmg_binary = ((mask_full == 2) | (mask_full == 3)).astype(np.uint8) * 255
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
dmg_clean = cv2.morphologyEx(cv2.morphologyEx(dmg_binary, cv2.MORPH_CLOSE, kernel), cv2.MORPH_OPEN, kernel)

contours, _ = cv2.findContours(dmg_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

quad_names = [
    ["Northwest", "North", "Northeast"],
    ["West",      "Center", "East"],
    ["Southwest", "South", "Southeast"]
]

detected_boxes = []
quad_counts = {}

for idx, c in enumerate(contours, start=1):
    area = cv2.contourArea(c)
    if area >= 50:  # noise filter
        x, y, bw, bh = cv2.boundingRect(c)
        cx, cy = x + bw / 2.0, y + bh / 2.0
        
        # 3x3 Quadrant mapping
        col = min(2, max(0, int(cx / (w / 3.0))))
        row = min(2, max(0, int(cy / (h / 3.0))))
        quad = quad_names[row][col]
        quad_counts[quad] = quad_counts.get(quad, 0) + 1
        
        # TEOChat normalized box [ymin, xmin, ymax, xmax] (0-1000 scale)
        norm_box = [int((y / h) * 1000), int((x / w) * 1000), int(((y + bh) / h) * 1000), int(((x + bw) / w) * 1000)]
        detected_boxes.append((idx, norm_box, quad, int(area)))

damaged_count = len(detected_boxes)
most_affected = max(quad_counts.items(), key=lambda x: x[1])[0] if quad_counts else "None"

# 5. Format the TEOChat Prompt
boxes_text = "\n".join([f"  - Building {b[0]}: location {b[1]} in {b[2]} (Area: {b[3]}px)" for b in detected_boxes[:6]])

vlm_augmented_prompt = f"""[High-Resolution U-Net Damage Findings]:
- Total Damaged/Destroyed Buildings Detected: {damaged_count}
- Primary Hazard Concentration Sector: {most_affected}
- Detected Building Coordinates [ymin, xmin, ymax, xmax]:
{boxes_text}

--- Question ---
How many buildings in the image have been damaged or destroyed?
Options:
A. 1
B. 3
C. 8
D. 14
Please output only the single best choice (e.g., 'The answer is B.') and explain your count."""

print("\n" + "=" * 60)
print("EXTRACTED FINDINGS:")
print(f"• Damaged Building Count: {damaged_count}")
print(f"• Most Affected Sector:   {most_affected}")
print("=" * 60)
print("\nGENERATED PROMPT FOR VLM:\n")
print(vlm_augmented_prompt)

best_model.pth: reconstructing file:   0%|          |  0.00B / 97.9MB            

best_model.pth: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

✓ Successfully loaded Champion U-Net weights!

EXTRACTED FINDINGS:
• Damaged Building Count: 24
• Most Affected Sector:   North

GENERATED PROMPT FOR VLM:

[High-Resolution U-Net Damage Findings]:
- Total Damaged/Destroyed Buildings Detected: 24
- Primary Hazard Concentration Sector: North
- Detected Building Coordinates [ymin, xmin, ymax, xmax]:
  - Building 3: location [929, 244, 992, 340] in Southwest (Area: 3492px)
  - Building 4: location [898, 0, 941, 31] in Southwest (Area: 819px)
  - Building 6: location [847, 835, 894, 908] in Southeast (Area: 2490px)
  - Building 9: location [636, 0, 991, 623] in Southwest (Area: 102486px)
  - Building 10: location [527, 679, 538, 696] in East (Area: 107px)
  - Building 13: location [483, 709, 501, 725] in East (Area: 207px)

--- Question ---
How many buildings in the image have been damaged or destroyed?
Options:
A. 1
B. 3
C. 8
D. 14
Please output only the single best choice (e.g., 'The answer is B.') and explain your count.


## 3. Unload U-Net to Free VRAM
Frees all 16 GB of VRAM on the T4 GPU before loading the 7B VLM.

In [3]:
# ── Cell 3: Clear VRAM ──
import gc

del unet_model
gc.collect()
torch.cuda.empty_cache()
print("✓ U-Net unloaded. 16 GB VRAM is now completely free for Qwen2.5-VL!")

✓ U-Net unloaded. 16 GB VRAM is now completely free for Qwen2.5-VL!


## 4. Query Qwen2.5-VL with Injected Structural Findings
Loads the fused FP16 Qwen2.5-VL model, consumes the image alongside the injected U-Net text findings, and generates the final count.

In [4]:
# ── Cell 4: Run Qwen2.5-VL with Injected Findings ──
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

model_id = "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP"

print(f"Loading VLM: {model_id}...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)

# Prepare multimodal input
image = Image.open(sample_img_path)
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": vlm_augmented_prompt},
        ],
    }
]

prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[prompt], images=[image], return_tensors="pt").to("cuda")

# Generate response
print("\nGenerating VLM reasoning and answer...")
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)

response = processor.decode(generated_ids[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

print("\n" + "=" * 60)
print("FINAL VLM ANSWER:")
print("=" * 60)
print(response)
print("=" * 60)

Loading VLM: AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP...


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]


Generating VLM reasoning and answer...

FINAL VLM ANSWER:
The total number of buildings that have been damaged or destroyed in the image is 24.
